# Mitra Classifier — End-to-End Classification with Your Own Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mitra-classifier-pipeline/blob/main/tutorials/mitra_classifier_colab.ipynb)

Use the **Mitra Classifier** weights distributed through the DIMER Model Repository with your own tabular data. Upload the DIMER ZIP when available, or use the exact pinned upstream checkpoint as a fallback; then evaluate pretrained Mitra, optionally fine-tune on a GPU, classify new rows, and export the predictor.

**No DIMER Workbench access is required.** Your data is processed in Google Colab, not by DIMER. Do not upload confidential, sensitive, or restricted data unless that environment is permitted.

## About this model

Mitra is a Transformer-based tabular foundation model from the AutoGluon team at AWS, pretrained on about **45 million synthetic tabular datasets** with no reported real-world pretraining data. This classifier uses `dim=512`, `dim_output=10`, `n_layers=12`, `n_heads=4`, `task=CLASSIFICATION` and contains about **75.7M parameters**. It accepts numerical and categorical features; target labels come from **your dataset**, not a fixed pretrained vocabulary.

The DIMER model card reports **0.858 ± 0.143 mean accuracy** across 137 benchmark datasets for **MITRA (+ef)**, which combines **ensembling (+e)** and **fine-tuning (+f)**. This is not guaranteed zero-shot accuracy or an expected score for your dataset; use the holdout evaluation below for your task.


## 1. Install the runtime

The `mitra` extra is required; plain `autogluon.tabular` does not include all Mitra runtime dependencies. PyTorch is left to the Colab runtime so its CUDA build stays compatible with the selected accelerator.


In [ ]:
%pip install -q "autogluon.tabular[mitra]==1.5.0"


## 2. Acquire and verify the checkpoint

DIMER hosts `model.safetensors`; the associated `config.json` is required to reconstruct the architecture.

- **DIMER ZIP** — extract the weights from DIMER and fetch the matching pinned config.
- **Pinned upstream** — fetch both files from the exact upstream revision when DIMER download is unavailable.

Both files are SHA-256 verified, installed into an isolated Hugging Face cache, then used offline so AutoGluon cannot silently resolve a newer checkpoint.


In [ ]:
import hashlib, json, os, random, shutil, urllib.request, zipfile
from pathlib import Path

HF_HOME = Path("/content/mitra-hf")
os.environ["HF_HOME"] = str(HF_HOME)

MODEL_ID = "autogluon/mitra-classifier"
PINNED_REVISION = "c425e9fa0910a6be1c494321792e7ba2a1367b1a"
EXPECTED_WEIGHTS_SHA256 = "e06a055e91a3baeffc37f9cf634d9e69a27d904b6686131dc3b702f9c0126b19"
EXPECTED_CONFIG_SHA256 = "2c96c24dd25f64e92753f6f2ba00cc7833b9923459403dcd8504e8700c0995df"

MODEL_DIR = Path("/content/mitra-model")
MODEL_DIR.mkdir(exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def download_pinned(name, dest):
    url = f"https://huggingface.co/{MODEL_ID}/resolve/{PINNED_REVISION}/{name}?download=true"
    print(f"Retrieving pinned {name}...")
    with urllib.request.urlopen(url) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)

def verify(path, expected, label):
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(
            f"{label} checksum mismatch.\nExpected: {expected}\nActual:   {actual}"
        )
    print(f"✓ {label} verified: {actual[:12]}…")

def weights_from_upload(dest):
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload exactly one DIMER ZIP or model.safetensors.")
    p = Path("/content") / next(iter(uploaded))

    if p.suffix.lower() == ".safetensors":
        shutil.copy2(p, dest)
        return

    if p.suffix.lower() != ".zip":
        raise ValueError("Expected a DIMER ZIP or model.safetensors.")

    with zipfile.ZipFile(p) as z:
        matches = [
            i for i in z.infolist()
            if not i.is_dir() and Path(i.filename).name == "model.safetensors"
        ]
        if len(matches) != 1:
            raise RuntimeError(
                f"Expected one model.safetensors in the DIMER ZIP; found {len(matches)}."
            )
        with z.open(matches[0]) as src, open(dest, "wb") as dst:
            shutil.copyfileobj(src, dst)

def install_offline_snapshot(weights, config):
    snapshot_id = sha256_file(weights)[:40]
    repo = HF_HOME / "hub" / ("models--" + MODEL_ID.replace("/", "--"))
    snap = repo / "snapshots" / snapshot_id
    refs = repo / "refs"
    snap.mkdir(parents=True, exist_ok=True)
    refs.mkdir(parents=True, exist_ok=True)

    shutil.copy2(weights, snap / "model.safetensors")
    shutil.copy2(config, snap / "config.json")
    (refs / "main").write_text(snapshot_id)

    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    return snap

MODEL_SOURCE = "Pinned upstream"  # @param ["DIMER ZIP", "Pinned upstream"]

weights_path = MODEL_DIR / "model.safetensors"
config_path = MODEL_DIR / "config.json"

if MODEL_SOURCE == "DIMER ZIP":
    weights_from_upload(weights_path)
    download_pinned("config.json", config_path)
else:
    download_pinned("model.safetensors", weights_path)
    download_pinned("config.json", config_path)

verify(weights_path, EXPECTED_WEIGHTS_SHA256, "model.safetensors")
verify(config_path, EXPECTED_CONFIG_SHA256, "config.json")

config = json.loads(config_path.read_text())
print("Model configuration:")
print(json.dumps(config, indent=2))
print(f"✓ Offline snapshot: {install_offline_snapshot(weights_path, config_path)}")


## 3. Bring your own data

Upload one CSV: rows are observations, one categorical column is the target, and the remaining retained columns are numerical or categorical features. Target values become the output classes.

Hard limits: **2–10 classes**, **≤500 features**, **≤10,000 training rows**. Mitra's particularly strong reported regime is about **≤5,000 samples and ≤100 features**; this notebook warns when you are within the hard limits but outside that regime.

Raw images, text, audio, or video must first be transformed into meaningful tabular features. The checks here are educational notebook checks, not DIMER Workbench validation.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

DATA_SOURCE = "Built-in demo"  # @param ["Built-in demo", "Upload CSV"]
TARGET_COLUMN = "target"       # @param {type:"string"}
DROP_COLUMNS = ""              # @param {type:"string"}
VALIDATION_SPLIT = 0.20        # @param {type:"number"}
SEED = 42                      # @param {type:"integer"}

if DATA_SOURCE == "Built-in demo":
    data = load_breast_cancer(as_frame=True).frame.copy()
    TARGET_COLUMN = "target"
else:
    from google.colab import files
    uploaded = files.upload()
    names = [n for n in uploaded if n.lower().endswith(".csv")]
    if len(names) != 1:
        raise RuntimeError("Upload exactly one CSV.")
    data = pd.read_csv(Path("/content") / names[0])

drop_columns = [c.strip() for c in DROP_COLUMNS.split(",") if c.strip() and c.strip() != TARGET_COLUMN]
if data.columns.duplicated().any():
    raise ValueError("Duplicate column names are not supported.")
if TARGET_COLUMN not in data.columns:
    raise ValueError(f"Target column {TARGET_COLUMN!r} not found.")

clean = data.drop(columns=[c for c in drop_columns if c in data.columns], errors="ignore").dropna(subset=[TARGET_COLUMN]).copy()
features = [c for c in clean.columns if c != TARGET_COLUMN]
counts = clean[TARGET_COLUMN].value_counts()
n_classes = len(counts)

errors = []
if len(clean) < 50:
    errors.append("Use at least 50 labelled rows.")
if not features:
    errors.append("No feature columns remain.")
if len(features) > 500:
    errors.append(f"{len(features)} features exceed Mitra's 500-feature limit.")
if not 2 <= n_classes <= 10:
    errors.append(f"Target has {n_classes} classes; Mitra requires 2–10.")
if len(counts) and counts.min() < 2:
    errors.append(f"Every class needs at least 2 rows; counts={counts.to_dict()}.")
if errors:
    raise ValueError("Dataset is not ready:\n- " + "\n- ".join(errors))
if not 0.05 <= VALIDATION_SPLIT <= 0.40:
    raise ValueError("VALIDATION_SPLIT must be 0.05–0.40.")

display(pd.DataFrame({"Item":["Usable rows","Features","Target","Classes"],"Value":[len(clean),len(features),TARGET_COLUMN,n_classes]}))
display(counts.rename("rows").to_frame())
if len(clean)>10_000:
    print("⚠ Above 10,000 rows; training will be class-preserving capped.")
elif len(clean)>5_000:
    print("⚠ Within the hard limit but above the particularly strong reported ≤5,000-sample regime.")
if len(features)>100:
    print("⚠ Within the hard limit but above the particularly strong reported ≤100-feature regime.")
if len(counts) and counts.min()/counts.sum()<0.05:
    print("⚠ Strong class imbalance: accuracy alone may mislead.")
near_unique=[c for c in features if clean[c].nunique(dropna=False)/len(clean)>0.98]
if near_unique:
    print("⚠ Nearly unique/identifier-like columns:", near_unique[:10])

train_data,holdout_data=train_test_split(clean,test_size=VALIDATION_SPLIT,random_state=SEED,stratify=clean[TARGET_COLUMN])
def stratified_cap(df, ceiling=10_000):
    if len(df)<=ceiling:
        return df.reset_index(drop=True)
    rng=np.random.RandomState(SEED); keep=[]
    for cls in df[TARGET_COLUMN].drop_duplicates():
        keep.append(rng.choice(df.index[df[TARGET_COLUMN]==cls].to_numpy()))
    keep_set=set(keep)
    remaining=np.array([i for i in df.index if i not in keep_set])
    keep.extend(rng.choice(remaining,size=ceiling-len(keep),replace=False))
    return df.loc[keep].sample(frac=1,random_state=SEED).reset_index(drop=True)

train_data=stratified_cap(train_data)
FEATURE_COLUMNS=[c for c in train_data.columns if c!=TARGET_COLUMN]
NUM_CLASSES=train_data[TARGET_COLUMN].nunique()
PROBLEM_TYPE="binary" if NUM_CLASSES==2 else "multiclass"
print(f"✓ train={len(train_data):,}, holdout={len(holdout_data):,}, features={len(FEATURE_COLUMNS)}, type={PROBLEM_TYPE}")


## 4. Evaluate pretrained Mitra, then optionally fine-tune

`fine_tune=False` uses labelled examples as context without updating Mitra's weights. Fine-tuning updates the weights and requires a GPU here; compare it against the same holdout baseline.

Accuracy can hide minority-class failures. Depending on the task, consider **balanced accuracy, macro F1, MCC, class-specific precision/recall, ROC-AUC or PR-AUC, and log loss**. If probabilities drive decisions, assess **calibration** separately.

The published 85.8% is for the benchmark **+ef** configuration, not standalone zero-shot checkpoint performance.

**Colab memory note:** Mitra's estimated peak RAM can sit just above AutoGluon's default 90% safety threshold on standard Colab runtimes. This tutorial sets `MAX_MEMORY_USAGE_RATIO=1.10`, which raises the guard slightly while preserving a safety margin. Values above `1.0` increase out-of-memory risk; reduce the dataset or use a higher-memory runtime if fitting still fails.


In [ ]:
import torch
import random
from autogluon.tabular import TabularPredictor

EVAL_METRIC = "accuracy"          # @param ["accuracy", "balanced_accuracy", "log_loss", "f1_macro", "mcc"]
BASELINE_TIME_LIMIT = 300         # @param {type:"integer"}
RUN_FINE_TUNING = False           # @param {type:"boolean"}
FINE_TUNE_STEPS = 0               # @param {type:"integer"}
FINE_TUNE_TIME_LIMIT = 600        # @param {type:"integer"}
MAX_MEMORY_USAGE_RATIO = 1.10     # @param {type:"number"}

CUDA_AVAILABLE = torch.cuda.is_available()
print("CUDA available:",CUDA_AVAILABLE,torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "")
print(f"AutoGluon memory safety ratio: {MAX_MEMORY_USAGE_RATIO:.2f}")

def seed_everything():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def fit_mitra(fine_tune,path,time_limit,fine_tune_steps=0):
    seed_everything(); hp={"fine_tune":fine_tune,"seed":SEED}
    if EVAL_METRIC in {"accuracy","log_loss"}: hp["metric"]=EVAL_METRIC
    if fine_tune and fine_tune_steps>0: hp["fine_tune_steps"]=fine_tune_steps
    predictor=TabularPredictor(label=TARGET_COLUMN,problem_type=PROBLEM_TYPE,eval_metric=EVAL_METRIC,path=path,verbosity=2)
    predictor.fit(
        train_data,
        hyperparameters={"MITRA":hp},
        fit_weighted_ensemble=False,
        time_limit=time_limit,
        ag_args_fit={"max_memory_usage_ratio": MAX_MEMORY_USAGE_RATIO},
    )
    if not any("mitra" in n.lower() for n in predictor.model_names()):
        raise RuntimeError(f"Expected Mitra; AutoGluon trained {predictor.model_names()}.")
    return predictor

def evaluate(predictor):
    raw=predictor.evaluate(holdout_data,auxiliary_metrics=True,silent=True)
    return {k:float(-v if "log_loss" in k else v) for k,v in raw.items()}

baseline_predictor=fit_mitra(False,"/content/mitra-baseline",BASELINE_TIME_LIMIT)
baseline_metrics=evaluate(baseline_predictor); display(pd.Series(baseline_metrics,name="Pretrained").to_frame())

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
pred=baseline_predictor.predict(holdout_data.drop(columns=[TARGET_COLUMN]))
ConfusionMatrixDisplay.from_predictions(holdout_data[TARGET_COLUMN],pred)
plt.title("Pretrained Mitra — holdout confusion matrix"); plt.show()

finetuned_predictor=finetuned_metrics=None
if RUN_FINE_TUNING:
    if not CUDA_AVAILABLE:
        raise RuntimeError("Fine-tuning requires a GPU. Choose Runtime → Change runtime type → GPU and rerun.")
    finetuned_predictor=fit_mitra(True,"/content/mitra-finetuned",FINE_TUNE_TIME_LIMIT,FINE_TUNE_STEPS)
    finetuned_metrics=evaluate(finetuned_predictor)
    keys=sorted(set(baseline_metrics)&set(finetuned_metrics))
    comparison=pd.DataFrame({"Pretrained":[baseline_metrics[k] for k in keys],"Fine-tuned":[finetuned_metrics[k] for k in keys]},index=keys)
    display(comparison)
else:
    print("Fine-tuning skipped. Set RUN_FINE_TUNING=True on a GPU to run it.")


## 5. Classify new rows

Upload a CSV with the same feature columns and no target column. The notebook reports predictions and class probabilities and lets you download `predictions.csv`.


In [ ]:
RUN_NEW_DATA_INFERENCE = False  # @param {type:"boolean"}

if RUN_NEW_DATA_INFERENCE:
    from google.colab import files
    uploaded = files.upload()
    names = [n for n in uploaded if n.lower().endswith(".csv")]
    if len(names) != 1:
        raise RuntimeError("Upload exactly one inference CSV.")
    new_data = pd.read_csv(Path("/content") / names[0])
    missing = [c for c in FEATURE_COLUMNS if c not in new_data.columns]
    if missing:
        raise ValueError(f"Inference CSV is missing required features: {missing}")
    X_new = new_data[FEATURE_COLUMNS].copy()
    active = finetuned_predictor or baseline_predictor
    pred = active.predict(X_new)
    proba = active.predict_proba(X_new)
    out = new_data.copy()
    out["prediction"] = pred.values
    for col in proba.columns:
        out[f"probability_{col}"] = proba[col].values
    out_path = "/content/predictions.csv"
    out.to_csv(out_path,index=False)
    display(out.head())
    files.download(out_path)
else:
    print("Inference skipped. Set RUN_NEW_DATA_INFERENCE=True when you have new rows to classify.")


## 6. Export the reusable predictor

The AutoGluon `TabularPredictor` directory is the reusable trained artifact. The cell below packages the active predictor with a compact run-metadata record.


In [ ]:
import json, shutil
from pathlib import Path

active_predictor = finetuned_predictor or baseline_predictor
active_path = Path(active_predictor.path)
metadata = {
    "base_model": MODEL_ID,
    "base_model_revision": PINNED_REVISION,
    "weights_sha256": EXPECTED_WEIGHTS_SHA256,
    "config_sha256": EXPECTED_CONFIG_SHA256,
    "autogluon_version": "1.5.0",
    "mode": "fine-tuned" if finetuned_predictor is not None else "pretrained",
    "target_column": TARGET_COLUMN,
    "features": FEATURE_COLUMNS,
    "seed": SEED,
    "ai_assistance": {
        "client": "OpenAI ChatGPT",
        "agent_relay_role": "Builder",
        "note": "Attribution is provenance, not sign-off or independent verification.",
    },
}
(active_path / "tutorial_run_metadata.json").write_text(json.dumps(metadata,indent=2))
archive=shutil.make_archive("/content/mitra-predictor","zip",root_dir=active_path)
print("✓ Predictor archive:",archive)


## AI use and provenance

This standalone tutorial was developed with substantial AI assistance from **OpenAI ChatGPT** under human direction and review. The maintainer defined the goal, scope, DIMER constraints, model release, and acceptance criteria and remains responsible for repository changes and release decisions.

- Generated with AI assistance by: **OpenAI ChatGPT**
- Agent Relay role: **Builder**
- Base-model developer: **AutoGluon team, Amazon Web Services (AWS)**
- DIMER role: distributor of the pinned `model.safetensors` artifact, not model developer
- Model identity: pinned upstream revision and SHA-256 values recorded above

AI attribution is **provenance, not sign-off**. It does not authenticate authorship, imply endorsement by OpenAI, AWS, AutoGluon, or DIMER, or independently verify correctness. Executed checks and reproducible outputs remain the evidence for a particular run.
